# Dataset Pipeline: TF-IDF vs. Hugging Face sentence embeddings
**Author:** Piper

Same two-part structure as the course setup notebook (install, token, connect to a model), but the Hugging Face model is used to **embed** comments, not to prompt. There is no zero-, one- or few-shot prompting anywhere.

Models compared (all one-vs-rest logistic regression, class-weighted, per-class thre
sholds tuned on validation):
* Baselines: neutral-only, label-frequency
* **TF-IDF:** (a) word, (b) word+char, (c) word+char+handcrafted, plus case-kept ablation
* **Embeddings:** (d) frozen sentence embeddings, (e) word TF-IDF + embeddings

**Model selection uses VALIDATION only.** The test set is scored once per model and never used to choose anything. Differences are reported with bootstrap confidence intervals.

**Runtime:** use a GPU (Runtime > Change runtime type > T4 GPU). Embedding on CPU takes about 10+ minutes, on GPU about 1-2 minutes. Set `USE_DRIVE = True` below so embeddings and predictions survive a disconnect.

## Install libraries

In [ ]:
%%capture
!pip install -U langchain-core langchain-huggingface huggingface_hub sentence-transformers "datasets>=2.19" pyarrow

## Modeling code (`pipeline.py`)
This cell writes the shared modeling code to a file in the Colab session, so the notebook is fully self-contained.

In [ ]:
%%writefile pipeline.py
"""
pipeline.py: shared modeling code for the GoEmotions ambiguity audit (proposal Section 6).

Everything that fits something (vocabulary, scaler, model, thresholds) is fit on
TRAIN only, except thresholds, which are tuned on VALIDATION only.
The test split is never touched until final evaluation.
"""
import os
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MaxAbsScaler
from sklearn.metrics import f1_score, hamming_loss

SEED = 42
N_CLASSES = 28          # 27 emotions + neutral
NEUTRAL_IDX = 27

# Keeps punctuation as its own tokens ("!", "?", "...") and keeps 1-char words ("i", "a").
TOKEN_PATTERN = r"(?u)\b\w+\b|[^\w\s]"

_NEG = re.compile(r"\b(?:not|no|never|nobody|nothing|nowhere|neither|nor|none|cannot)\b|n't\b", re.I)

HAND_NAMES = ["n_chars", "n_words", "n_exclaim", "n_question", "n_ellipsis",
              "n_punct", "caps_ratio", "n_negation"]



# Data
def multi_hot(label_lists, n=N_CLASSES):
    Y = np.zeros((len(label_lists), n), dtype=np.int8)
    for i, ls in enumerate(label_lists):
        Y[i, list(ls)] = 1
    return Y


def load_simplified():
    """Returns (dict of split -> DataFrame[comment_id, text, labels], label_names)."""
    from datasets import load_dataset
    try:
        ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    except Exception:
        ds = load_dataset("go_emotions", "simplified")
    names = ds["train"].features["labels"].feature.names
    assert len(names) == N_CLASSES and names[NEUTRAL_IDX] == "neutral", names
    out = {}
    for split in ["train", "validation", "test"]:
        df = ds[split].to_pandas()
        if "id" in df.columns:
            df = df.rename(columns={"id": "comment_id"})
        df["split"] = split
        out[split] = df[["comment_id", "split", "text", "labels"]].reset_index(drop=True)
    return out, names


# Features
def handcrafted(texts):
    rows = []
    for t in texts:
        letters = [c for c in t if c.isalpha()]
        caps = sum(c.isupper() for c in letters) / max(len(letters), 1)
        n_punct = sum((not c.isalnum()) and (not c.isspace()) for c in t)
        rows.append([len(t), len(t.split()), t.count("!"), t.count("?"),
                     t.count("..."), n_punct, caps, len(_NEG.findall(t))])
    return np.log1p(np.asarray(rows, dtype=np.float32))


class FeatureBuilder:
    """
    word : word 1-2-gram TF-IDF
    char : char_wb 2-5-gram TF-IDF
    hand : handcrafted features (length, punctuation, caps ratio, negation cues)
    lowercase=False gives the case-kept ablation.
    Variants from the proposal: (a) word  (b) word+char  (c) word+char+hand
    """
    def __init__(self, word=True, char=False, hand=False, lowercase=True,
                 char_max_features=200_000):
        self.word, self.char, self.hand = word, char, hand
        self.lowercase = lowercase
        self.char_max_features = char_max_features

    def fit(self, texts):
        self.parts_ = []
        if self.word:
            self.wv_ = TfidfVectorizer(lowercase=self.lowercase, ngram_range=(1, 2),
                                       token_pattern=TOKEN_PATTERN, min_df=2,
                                       sublinear_tf=True, stop_words=None,
                                       dtype=np.float32)
            self.wv_.fit(texts)
        if self.char:
            self.cv_ = TfidfVectorizer(lowercase=self.lowercase, analyzer="char_wb",
                                       ngram_range=(2, 5), min_df=3,
                                       max_features=self.char_max_features,
                                       sublinear_tf=True, dtype=np.float32)
            self.cv_.fit(texts)
        if self.hand:
            self.scaler_ = MaxAbsScaler().fit(handcrafted(texts))
        return self

    def transform(self, texts):
        mats = []
        if self.word:
            mats.append(self.wv_.transform(texts))
        if self.char:
            mats.append(self.cv_.transform(texts))
        if self.hand:
            mats.append(sp.csr_matrix(self.scaler_.transform(handcrafted(texts))))
        return sp.hstack(mats, format="csr", dtype=np.float32)

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)

    def feature_names(self):
        names = []
        if self.word:
            names += ["w:" + f for f in self.wv_.get_feature_names_out()]
        if self.char:
            names += ["c:" + f for f in self.cv_.get_feature_names_out()]
        if self.hand:
            names += ["h:" + f for f in HAND_NAMES]
        return np.array(names)


# Model
def train_ovr(X, Y, C=1.0, n_jobs=-1):
    """One-vs-rest logistic regression. liblinear for sparse TF-IDF; lbfgs for dense
    embeddings (about 4x faster there, same objective up to the intercept penalty)."""
    solver = "liblinear" if sp.issparse(X) else "lbfgs"
    clf = OneVsRestClassifier(
        LogisticRegression(C=C, class_weight="balanced", solver=solver,
                           max_iter=200 if solver == "liblinear" else 500,
                           random_state=SEED),
        n_jobs=n_jobs)
    return clf.fit(X, Y)


def top_features(clf, feat_names, class_idx, k=15):
    """Top positive-weight features for one class (used to explain predictions)."""
    est = clf.estimators_[class_idx]
    if not hasattr(est, "coef_"):
        return []
    w = est.coef_.ravel()
    idx = np.argsort(w)[::-1][:k]
    return [(feat_names[i], float(w[i])) for i in idx]


def oof_probs(texts, Y, builder_kwargs, C=1.0, n_splits=5):
    """Out-of-fold probabilities. The vectorizer is refit inside every fold (no leakage)."""
    texts = np.asarray(texts, dtype=object)
    P = np.zeros(Y.shape, dtype=np.float32)
    for tr, te in KFold(n_splits, shuffle=True, random_state=SEED).split(texts):
        fb = FeatureBuilder(**builder_kwargs)
        Xtr = fb.fit_transform(list(texts[tr]))
        clf = train_ovr(Xtr, Y[tr], C=C)
        P[te] = clf.predict_proba(fb.transform(list(texts[te])))
    return P


# Pretrained sentence embeddings (Hugging Face)
def embed_texts(embedder, texts, chunk=2000, cache_path=None):
    """embedder: any object with .embed_documents(list[str]) (e.g. langchain_huggingface.HuggingFaceEmbeddings).
    Embeds in chunks and caches to .npy so a Colab disconnect doesn't cost you the run."""
    if cache_path and os.path.exists(cache_path):
        return np.load(cache_path)
    texts = list(texts)
    out = [np.asarray(embedder.embed_documents(texts[i:i + chunk]), dtype=np.float32)
           for i in range(0, len(texts), chunk)]
    E = np.vstack(out)
    if cache_path:
        np.save(cache_path, E)
    return E


def with_embeddings(X_sparse, E):
    """Append dense embeddings to a sparse TF-IDF matrix (combined variant)."""
    return sp.hstack([X_sparse, sp.csr_matrix(E)], format="csr", dtype=np.float32)


def oof_probs_features(X, Y, C=1.0, n_splits=5):
    """Out-of-fold probabilities for PRECOMPUTED features (e.g. embeddings; nothing is fit
    on text, so there is no vocabulary leakage to worry about)."""
    P = np.zeros(Y.shape, dtype=np.float32)
    for tr, te in KFold(n_splits, shuffle=True, random_state=SEED).split(np.arange(Y.shape[0])):
        clf = train_ovr(X[tr], Y[tr], C=C)
        P[te] = clf.predict_proba(X[te])
    return P


# Thresholds and metrics
GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)


def tune_thresholds(P, Y, grid=GRID):
    """Per-class threshold maximizing F1. Call with VALIDATION data only."""
    th = np.full(P.shape[1], 0.5)
    for j in range(P.shape[1]):
        y = Y[:, j].astype(bool)
        if y.sum() == 0:
            continue
        pred = P[:, j, None] >= grid[None, :]
        tp = (pred & y[:, None]).sum(0)
        denom = pred.sum(0) + y.sum()
        f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
        th[j] = grid[int(np.argmax(f1))]
    return th


def apply_thresholds(P, th, force_one=False):
    """force_one=True: if no class clears its threshold, predict the class with the
    highest P/threshold ratio (every GoEmotions comment has at least one gold label)."""
    pred = (P >= th[None, :]).astype(np.int8)
    if force_one:
        empty = pred.sum(1) == 0
        if empty.any():
            best = np.argmax(P[empty] / th[None, :], axis=1)
            pred[np.where(empty)[0], best] = 1
    return pred


def evaluate(Y, pred, names):
    per_class = f1_score(Y, pred, average=None, zero_division=0)
    return {
        "macro_f1": f1_score(Y, pred, average="macro", zero_division=0),
        "micro_f1": f1_score(Y, pred, average="micro", zero_division=0),
        "hamming": hamming_loss(Y, pred),
        "per_class_f1": dict(zip(names, np.round(per_class, 4))),
    }


# Baselines (No text used)
def neutral_only(n_rows):
    P = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    P[:, NEUTRAL_IDX] = 1.0
    return P, (P >= 0.5).astype(np.int8)


def label_frequency(n_rows, Y_train, seed=SEED):
    """Predicts each label independently with probability = its train prevalence.
    The 'probability' output is the constant prevalence vector."""
    prev = Y_train.mean(0).astype(np.float32)
    P = np.tile(prev, (n_rows, 1))
    rng = np.random.default_rng(seed)
    pred = (rng.random(P.shape) < P).astype(np.int8)
    return P, pred


# Exports for the evluation notebook
def predictions_frame(comment_ids, split, Y, P, pred, names):
    df = pd.DataFrame({"comment_id": list(comment_ids), "split": list(split)})
    cols = {}
    for j, n in enumerate(names):
        cols[f"gold_{n}"] = Y[:, j].astype(np.int8)
        cols[f"prob_{n}"] = P[:, j].astype(np.float32)
        cols[f"pred_{n}"] = pred[:, j].astype(np.int8)
    return pd.concat([df, pd.DataFrame(cols)], axis=1)


# Bootstrap confidence intervals (paired: every model sees the same resamples)
def bootstrap_macro_f1(Y, preds, n_boot=1000, seed=SEED):
    """preds: dict tag -> 0/1 prediction matrix on the same rows as Y.
    Returns dict tag -> array of n_boot macro-F1 values. Because the resamples are shared,
    out[a] - out[b] is a valid paired difference."""
    rng = np.random.default_rng(seed)
    Yb_all = Y.astype(bool)
    P = {k: v.astype(bool) for k, v in preds.items()}
    N = Y.shape[0]
    out = {k: np.empty(n_boot) for k in P}
    for b in range(n_boot):
        idx = rng.integers(0, N, N)
        y = Yb_all[idx]
        for k, p in P.items():
            pb = p[idx]
            tp = (y & pb).sum(0)
            d = 2 * tp + (~y & pb).sum(0) + (y & ~pb).sum(0)
            out[k][b] = np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0).mean()
    return out


def ci95(x):
    lo, hi = np.percentile(x, [2.5, 97.5])
    return float(lo), float(hi)


Writing pipeline.py


In [ ]:
import os, getpass, time
import numpy as np, pandas as pd
import sklearn, datasets, torch
from langchain_huggingface import HuggingFaceEmbeddings
import importlib, pipeline as pl
importlib.reload(pl)

print("sklearn", sklearn.__version__, "| datasets", datasets.__version__)
HAS_GPU = torch.cuda.is_available()
print("GPU available:", HAS_GPU, "" if HAS_GPU else "  <-- embeddings will be SLOW. Runtime > Change runtime type > GPU, then rerun.")

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = "/content/drive/MyDrive/cs74_outputs"
else:
    OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)
print("saving to", OUT)

sklearn 1.6.1 | datasets 5.2.0
GPU available: False   <-- embeddings will be SLOW. Runtime > Change runtime type > GPU, then rerun.
saving to /content/outputs


## Hugging Face access token
Create a token at **huggingface.co > Settings > Access Tokens** (read access is enough). The cell asks for it when it runs, so it is never saved in the notebook. **Do not paste your token into a code cell you share or submit.**

The embedding model below is public, so the token is optional (it mainly avoids download rate limits). Press Enter to skip.

In [ ]:
if "HF_TOKEN" not in os.environ and "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    tok = getpass.getpass("Paste your Hugging Face token (or press Enter to skip): ")
    if tok:
        os.environ["HUGGINGFACEHUB_API_TOKEN"] = tok
if "HUGGINGFACEHUB_API_TOKEN" in os.environ:
    os.environ.setdefault("HF_TOKEN", os.environ["HUGGINGFACEHUB_API_TOKEN"])   # huggingface_hub reads HF_TOKEN

Paste your Hugging Face token (or press Enter to skip): ··········


## Connect to an embedding model
* `model_name`: any sentence-transformers model on the Hub. `all-MiniLM-L6-v2` (384-d) is small and fast; `all-mpnet-base-v2` (768-d) is stronger but slower.
* `normalize_embeddings=True` puts every vector on the unit sphere, which suits logistic regression.

This runs locally in Colab (no hosted-inference calls), so there are no API rate limits for 54K comments.

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"      # change to try another model; record which one you used
embedder = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs={"normalize_embeddings": True, "batch_size": 256 if HAS_GPU else 64},
)
v = embedder.embed_query("I freakin love this channel")
print(model_name, "->", len(v), "dimensions")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sentence-transformers/all-MiniLM-L6-v2 -> 384 dimensions


## Load simplified GoEmotions and sanity-check

In [ ]:
data, names = pl.load_simplified()
tr, va, te = data["train"], data["validation"], data["test"]
print({k: len(v) for k, v in data.items()})            # expect 43410 / 5426 / 5427

Ytr, Yva, Yte = (pl.multi_hot(d["labels"]) for d in (tr, va, te))
prev = pd.Series(Ytr.mean(0), index=names).sort_values(ascending=False)
print(prev.head(5).round(3).to_dict(), "...", prev.tail(3).round(4).to_dict())
assert not set(tr.comment_id) & set(te.comment_id), "split leakage"

# Near-duplicate check: identical comment text appearing in more than one split (case/whitespace-insensitive)
norm = lambda s: s.str.lower().str.strip()
train_texts = set(norm(tr.text))
te_dup = norm(te.text).isin(train_texts)
va_dup = norm(va.text).isin(train_texts)
print(f"test comments whose text also appears in train: {te_dup.sum()} ({te_dup.mean():.2%}); validation: {va_dup.sum()} ({va_dup.mean():.2%})")

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

{'train': 43410, 'validation': 5426, 'test': 5427}
{'neutral': 0.328, 'admiration': 0.095, 'approval': 0.068, 'gratitude': 0.061, 'annoyance': 0.057} ... {'relief': 0.0035, 'pride': 0.0026, 'grief': 0.0018}
test comments whose text also appears in train: 42 (0.77%); validation: 49 (0.90%)


## Helpers and baselines

In [ ]:
SPLITS = (("train", tr, Ytr), ("validation", va, Yva), ("test", te, Yte))
results, models = {}, {}

def export(tag, P, pred):
    df = pd.concat([pl.predictions_frame(d.comment_id, d.split, Y, P[s], pred[s], names)
                    for s, d, Y in SPLITS], ignore_index=True)
    df.to_parquet(f"{OUT}/predictions_{tag}.parquet", index=False)

def run_variant(tag, X, C=1.0, fb=None):
    """X: dict split -> feature matrix. Fits on train, tunes thresholds on validation, exports, scores.
    val_macro_f1 is what you use to CHOOSE between models; test numbers are for reporting only."""
    t0 = time.time()
    clf = pl.train_ovr(X["train"], Ytr, C=C)
    P = {s: clf.predict_proba(X[s]) for s in X}
    th = pl.tune_thresholds(P["validation"], Yva)                      # validation only
    pred = {s: pl.apply_thresholds(P[s], th) for s in P}
    export(tag, P, pred)
    r = pl.evaluate(Yte, pred["test"], names)
    val_f1 = pl.evaluate(Yva, pred["validation"], names)["macro_f1"]    # optimistic (thresholds tuned here) but equally so for every model
    r05 = pl.evaluate(Yte, pl.apply_thresholds(P["test"], np.full(28, 0.5)), names)
    r1 = pl.evaluate(Yte, pl.apply_thresholds(P["test"], th, force_one=True), names)
    results[tag] = dict(val_macro_f1=val_f1, macro_f1=r["macro_f1"], micro_f1=r["micro_f1"], hamming=r["hamming"],
                        macro_f1_thr0p5=r05["macro_f1"], macro_f1_force_one=r1["macro_f1"],
                        n_features=X["train"].shape[1], C=C, sec=round(time.time() - t0))
    models[tag] = dict(clf=clf, th=th, fb=fb, eval=r, pred_test=pred["test"])
    np.save(f"{OUT}/thresholds_{tag}.npy", th)
    print(tag, {k: round(v, 4) if isinstance(v, float) else v for k, v in results[tag].items()})

# Baselines: neutral-only and label-frequency (no text used)
for tag, fn in (("neutral_only", pl.neutral_only), ("label_frequency", lambda n: pl.label_frequency(n, Ytr))):
    P, pred = {}, {}
    for s, d, Y in SPLITS:
        P[s], pred[s] = fn(len(Y))
    export(tag, P, pred)
    r = pl.evaluate(Yte, pred["test"], names)
    results[tag] = {k: r[k] for k in ("macro_f1", "micro_f1", "hamming")}
    models[tag] = dict(pred_test=pred["test"])
pd.DataFrame(results).T

,macro_f1,micro_f1,hamming
neutral_only,0.017694,0.304015,0.053845
label_frequency,0.043818,0.130646,0.072955


## TF-IDF variants
(a) word 1-2-grams, (b) + char 2-5-grams, (c) + handcrafted features, and the case-kept ablation of (a). Vocabulary fit on train only.
Steps: build features, pick C on **validation** using variant (a), then fit every variant with that C.

In [ ]:
# Build features for every variant (fit on train only)
TFIDF_VARIANTS = {
    "a_word":           dict(word=True),
    "b_word_char":      dict(word=True, char=True),
    "c_word_char_hand": dict(word=True, char=True, hand=True),
    "a_word_casekept":  dict(word=True, lowercase=False),
}
sparse_X, fbs = {}, {}
for tag, kw in TFIDF_VARIANTS.items():
    t0 = time.time()
    fbs[tag] = pl.FeatureBuilder(**kw)
    Xs = {"train": fbs[tag].fit_transform(list(tr.text))}
    Xs["validation"], Xs["test"] = fbs[tag].transform(list(va.text)), fbs[tag].transform(list(te.text))
    sparse_X[tag] = Xs
    print(tag, Xs["train"].shape, f"{time.time() - t0:.0f}s")

a_word (43410, 65875) 6s
b_word_char (43410, 131588) 25s
c_word_char_hand (43410, 131596) 26s
a_word_casekept (43410, 69072) 4s


In [ ]:
# Small C sweep for TF-IDF on variant (a), chosen by VALIDATION macro-F1 (same treatment the embedding model gets)
# This C = 1 ultimately led to an inconclusive result in our report

rows = []
for C_ in (0.5, 1, 2):
    clf_ = pl.train_ovr(sparse_X["a_word"]["train"], Ytr, C=C_)
    Pv = clf_.predict_proba(sparse_X["a_word"]["validation"])
    th_ = pl.tune_thresholds(Pv, Yva)
    rows.append((C_, pl.evaluate(Yva, pl.apply_thresholds(Pv, th_), names)["macro_f1"]))
sweep_t = pd.DataFrame(rows, columns=["C", "val_macro_f1"]); display(sweep_t)
C_TFIDF = float(sweep_t.loc[sweep_t.val_macro_f1.idxmax(), "C"])
print("C_TFIDF =", C_TFIDF, "(if this is the smallest or largest value, extend the grid and rerun)")

,C,val_macro_f1
0,0.5,0.492743
1,1.0,0.495846
2,2.0,0.497381


C_TFIDF = 2.0 (if this is the smallest or largest value, extend the grid and rerun)


In [ ]:
# Fit and score every TF-IDF variant. Char variants take a few minutes each.
for tag in TFIDF_VARIANTS:
    run_variant(tag, sparse_X[tag], C=C_TFIDF, fb=fbs[tag])

a_word {'val_macro_f1': 0.4974, 'macro_f1': 0.4632, 'micro_f1': 0.5369, 'hamming': 0.0471, 'macro_f1_thr0p5': 0.4568, 'macro_f1_force_one': 0.4638, 'n_features': 65875, 'C': 2.0, 'sec': 19}
b_word_char {'val_macro_f1': 0.5054, 'macro_f1': 0.4711, 'micro_f1': 0.5476, 'hamming': 0.0446, 'macro_f1_thr0p5': 0.466, 'macro_f1_force_one': 0.4718, 'n_features': 131588, 'C': 2.0, 'sec': 127}
c_word_char_hand {'val_macro_f1': 0.5068, 'macro_f1': 0.4715, 'micro_f1': 0.5443, 'hamming': 0.0462, 'macro_f1_thr0p5': 0.4724, 'macro_f1_force_one': 0.4742, 'n_features': 131596, 'C': 2.0, 'sec': 159}
a_word_casekept {'val_macro_f1': 0.4914, 'macro_f1': 0.4492, 'micro_f1': 0.5253, 'hamming': 0.0483, 'macro_f1_thr0p5': 0.4492, 'macro_f1_force_one': 0.45, 'n_features': 69072, 'C': 2.0, 'sec': 17}


## Sentence-embedding variants
Embeddings come from a **frozen** pretrained model, so nothing is fitted on text and there is no vocabulary leakage. Caveat for the report: the pretrained model saw large amounts of web text during its own training, which may include Reddit. Say this in the limitations.

Embeddings are cached to `.npy` (in `OUT`), so rerunning this cell is instant if the files still exist.

In [ ]:
E = {}
for s, d, _ in SPLITS:
    t0 = time.time()
    E[s] = pl.embed_texts(embedder, d.text, cache_path=f"{OUT}/emb_{model_name.split('/')[-1]}_{s}.npy")
    print(s, E[s].shape, f"{time.time() - t0:.0f}s")

train (43410, 384) 520s
validation (5426, 384) 63s
test (5427, 384) 64s


In [ ]:
# sweep for the embedding model, chosen by VALIDATION macro-F1.
# The first run had its best score at C=1, the edge of the old grid, so the grid now extends below 1.
rows = []
for C_ in (0.1, 0.25, 0.5, 1, 2):
    clf_ = pl.train_ovr(E["train"], Ytr, C=C_)
    Pv = clf_.predict_proba(E["validation"])
    th_ = pl.tune_thresholds(Pv, Yva)
    rows.append((C_, pl.evaluate(Yva, pl.apply_thresholds(Pv, th_), names)["macro_f1"]))
sweep = pd.DataFrame(rows, columns=["C", "val_macro_f1"]); display(sweep)
C_EMB = float(sweep.loc[sweep.val_macro_f1.idxmax(), "C"])
print("C_EMB =", C_EMB, "(if this is the smallest or largest value, extend the grid and rerun)")

,C,val_macro_f1
0,0.10,0.377163
1,0.25,0.374103
2,0.50,0.369555
3,1.00,0.365111
4,2.00,0.359524


C_EMB = 0.1 (if this is the smallest or largest value, extend the grid and rerun)


In [ ]:
#  embeddings only, (e) word TF-IDF + embeddings
run_variant("d_embed", E, C=C_EMB)

Xcomb = {s: pl.with_embeddings(sparse_X["a_word"][s], E[s]) for s in E}
run_variant("e_word_plus_embed", Xcomb, C=C_EMB)

d_embed {'val_macro_f1': 0.3772, 'macro_f1': 0.3559, 'micro_f1': 0.4426, 'hamming': 0.0594, 'macro_f1_thr0p5': 0.2324, 'macro_f1_force_one': 0.3604, 'n_features': 384, 'C': 0.1, 'sec': 21}
e_word_plus_embed {'val_macro_f1': 0.4686, 'macro_f1': 0.4381, 'micro_f1': 0.5157, 'hamming': 0.0483, 'macro_f1_thr0p5': 0.331, 'macro_f1_force_one': 0.4394, 'n_features': 66259, 'C': 0.1, 'sec': 106}


## Compare (models are chosen on validation; test is for reporting)

In [ ]:
summary = pd.DataFrame(results).T
summary.to_csv(f"{OUT}/model_summary.csv")
cols = ["val_macro_f1", "macro_f1", "micro_f1", "hamming", "macro_f1_thr0p5", "macro_f1_force_one", "n_features", "C", "sec"]
display(summary[cols].round(4))

# Choose by VALIDATION macro-F1 (never by test)
best_tfidf = max(TFIDF_VARIANTS, key=lambda k: results[k]["val_macro_f1"])
best_emb = max(["d_embed", "e_word_plus_embed"], key=lambda k: results[k]["val_macro_f1"])
HEADLINE = best_tfidf       # the proposal audits the TF-IDF classifier; embeddings are a robustness comparison
print("best TF-IDF (by validation):", best_tfidf, "| best embedding model (by validation):", best_emb, "| HEADLINE:", HEADLINE)

,val_macro_f1,macro_f1,micro_f1,hamming,macro_f1_thr0p5,macro_f1_force_one,n_features,C,sec
neutral_only,NaN,0.0177,0.3040,0.0538,NaN,NaN,NaN,NaN,NaN
label_frequency,NaN,0.0438,0.1306,0.0730,NaN,NaN,NaN,NaN,NaN
a_word,0.4974,0.4632,0.5369,0.0471,0.4568,0.4638,65875.0,2.0,19.0
b_word_char,0.5054,0.4711,0.5476,0.0446,0.4660,0.4718,131588.0,2.0,127.0
c_word_char_hand,0.5068,0.4715,0.5443,0.0462,0.4724,0.4742,131596.0,2.0,159.0
a_word_casekept,0.4914,0.4492,0.5253,0.0483,0.4492,0.4500,69072.0,2.0,17.0
d_embed,0.3772,0.3559,0.4426,0.0594,0.2324,0.3604,384.0,0.1,21.0
e_word_plus_embed,0.4686,0.4381,0.5157,0.0483,0.3310,0.4394,66259.0,0.1,106.0


best TF-IDF (by validation): c_word_char_hand | best embedding model (by validation): e_word_plus_embed | HEADLINE: c_word_char_hand


In [ ]:
# 95% bootstrap CIs on TEST macro-F1 (same resamples for every model, so differences are paired)
fitted = [k for k in results if "pred_test" in models[k] and k not in ("neutral_only", "label_frequency")]
boot = pl.bootstrap_macro_f1(Yte, {k: models[k]["pred_test"] for k in fitted}, n_boot=1000)
ci_tbl = pd.DataFrame({k: dict(test_macro_f1=results[k]["macro_f1"], ci_low=pl.ci95(v)[0], ci_high=pl.ci95(v)[1]) for k, v in boot.items()}).T
display(ci_tbl.round(4))

def paired(a, b):
    lo, hi = pl.ci95(boot[a] - boot[b])
    verdict = "clear difference" if (lo > 0 or hi < 0) else "NOT distinguishable from zero"
    print(f"{a} minus {b}: {np.mean(boot[a] - boot[b]):+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  -> {verdict}")

paired(best_tfidf, best_emb)
paired("b_word_char", "a_word")
paired("a_word", "a_word_casekept")        # the case-kept ablation

,test_macro_f1,ci_low,ci_high
a_word,0.4632,0.4379,0.4829
b_word_char,0.4711,0.4474,0.4902
c_word_char_hand,0.4715,0.4460,0.4920
a_word_casekept,0.4492,0.4244,0.4677
d_embed,0.3559,0.3378,0.3707
e_word_plus_embed,0.4381,0.4151,0.4569


c_word_char_hand minus e_word_plus_embed: +0.0330  95% CI [+0.0162, +0.0485]  -> clear difference
b_word_char minus a_word: +0.0086  95% CI [-0.0061, +0.0220]  -> NOT distinguishable from zero
a_word minus a_word_casekept: +0.0137  95% CI [+0.0059, +0.0224]  -> clear difference


In [ ]:
# Per-class test F1: best TF-IDF vs best embedding model (rare classes are noisy; check test_support)
cmp = pd.DataFrame({best_tfidf: models[best_tfidf]["eval"]["per_class_f1"],
                    best_emb: models[best_emb]["eval"]["per_class_f1"]})
cmp["test_support"] = Yte.sum(0)
cmp["diff"] = cmp[best_emb] - cmp[best_tfidf]
display(cmp.sort_values("diff"))

# Duplicate sensitivity: does the headline score change if test comments seen in train are removed?
keep = ~te_dup.values
if te_dup.sum() > 0:
    r_full = pl.evaluate(Yte, models[HEADLINE]["pred_test"], names)["macro_f1"]
    r_dedup = pl.evaluate(Yte[keep], models[HEADLINE]["pred_test"][keep], names)["macro_f1"]
    print(f"{HEADLINE}: macro-F1 full test = {r_full:.4f}, test without train-duplicates = {r_dedup:.4f} (n={keep.sum()})")
else:
    print("No test comments duplicated in train.")

,c_word_char_hand,e_word_plus_embed,test_support,diff
embarrassment,0.4225,0.2712,37,-0.1513
joy,0.5815,0.4779,161,-0.1036
desire,0.4672,0.3742,83,-0.0930
fear,0.6170,0.5385,78,-0.0785
sadness,0.5098,0.4367,156,-0.0731
surprise,0.4658,0.3954,141,-0.0704
gratitude,0.9073,0.8441,352,-0.0632
realization,0.3072,0.2481,145,-0.0591
admiration,0.6448,0.5963,504,-0.0485
disgust,0.4523,0.4125,123,-0.0398


c_word_char_hand: macro-F1 full test = 0.4715, test without train-duplicates = 0.4660 (n=5385)


## Explanations
 Explanations use the word-only model to show top-weighted words (a_word), whose features are all readable. The headline model may use character n-grams, which produce fragments like `c: lov`. Embedding dimensions are not human-readable, so for the embedding model the audit relies on slice analysis and error review instead. Mention this asymmetry in the report.

In [ ]:
EXPLAIN = "a_word"
fb, clf, th = models[EXPLAIN]["fb"], models[EXPLAIN]["clf"], models[EXPLAIN]["th"]
fn_ = fb.feature_names()
pd.Series({names[j]: ", ".join(f for f, _ in pl.top_features(clf, fn_, j, 8)) for j in range(28)}).to_frame("top features")

,top features
admiration,"w:great, w:amazing, w:good, w:awesome, w:beaut..."
amusement,"w:lol, w:haha, w:funny, w:lmao, w:fun, w:hilar..."
anger,"w:fuck, w:hate, w:fucking, w:angry, w:stupid, ..."
annoyance,"w:annoying, w:stupid, w:dumb, w:annoyed, w:shi..."
approval,"w:agree, w:agreed, w:yes, w:i agree, w:right, ..."
caring,"w:concerned, w:worry, w:careful, w:you, w:help..."
confusion,"w:?, w:confused, w:not sure, w:idk, w:or, w:wh..."
curiosity,"w:?, w:curious, w:why, w:interesting, w:what, ..."
desire,"w:wish, w:want, w:i wish, w:need, w:wanted, w:..."
disappointment,"w:disappointed, w:disappointing, w:bad, w:upse..."


In [ ]:
def explain(comment_text, class_name, k=8):
    j = names.index(class_name)
    x = fb.transform([comment_text])
    contrib = x.multiply(clf.estimators_[j].coef_).tocoo()
    pairs = sorted(zip(contrib.col, contrib.data), key=lambda t: -abs(t[1]))[:k]
    print(f"[{EXPLAIN}] P({class_name}) = {clf.predict_proba(x)[0, j]:.3f}, threshold = {th[j]:.2f}")
    return [(fn_[c], round(float(v), 3)) for c, v in pairs]

explain("I freakin love this channel", "love")

[a_word] P(love) = 0.991, threshold = 0.68


[(np.str_('w:love'), 7.433),
 (np.str_('w:i'), 0.808),
 (np.str_('w:love this'), 0.667),
 (np.str_('w:i freakin'), -0.303),
 (np.str_('w:freakin'), -0.273),
 (np.str_('w:this'), 0.186),
 (np.str_('w:this channel'), -0.182),
 (np.str_('w:channel'), 0.028)]

## predictions over train + validation
About 49K comments with honest (not in-sample) probabilities for the ambiguity analysis. Embeddings are cheap to run out-of-fold (features are precomputed)

In [ ]:
trva = pd.concat([tr, va], ignore_index=True)
Ytrva = np.vstack([Ytr, Yva])
Etrva = np.vstack([E["train"], E["validation"]])
P_oof = pl.oof_probs_features(Etrva, Ytrva, C=C_EMB, n_splits=5)
th_oof = pl.tune_thresholds(P_oof, Ytrva)
pred_oof = pl.apply_thresholds(P_oof, th_oof)
pl.predictions_frame(trva.comment_id, trva.split, Ytrva, P_oof, pred_oof, names)\
  .to_parquet(f"{OUT}/predictions_d_embed_OOF.parquet", index=False)
print("OOF macro-F1:", pl.evaluate(Ytrva, pred_oof, names)["macro_f1"])

# TF-IDF OOF (slower; vocabulary is refit inside each fold):
# P_oof_t = pl.oof_probs(list(trva.text), Ytrva, TFIDF_VARIANTS["a_word"], C=C_TFIDF, n_splits=5)

OOF macro-F1: 0.3596315746297797
